# 04 · Data Warehouse
**CIS 517 – Hotel Booking Cancellation Prediction Using Data Mining**

Goal: load the cleaned bookings into a **star schema** (one fact table + six dimension tables),
then answer business questions with analytical SQL and OLAP operations (roll-up, drill-down, slice, dice, pivot, ranking).
The warehouse is built in **SQLite**, which is included with Python.

ــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــ


### 1. Dimensional model (design)

**Grain:** one row in the fact table = one booking (86,637 rows).

| Table | Type | Columns |
| --- | --- | --- |
| `Fact_Booking` | Fact | keys to all six dimensions + measures: lead_time, total_nights, adults, children, babies, adr, booked_revenue, realized_revenue, is_canceled, booking_changes, total_of_special_requests |
| `Dim_Date` | Dimension | date_key (YYYYMMDD), full_date, year, quarter, month, month_name, week_number, day_of_month, day_name, season, is_weekend |
| `Dim_Hotel` | Dimension | hotel_key, hotel_name |
| `Dim_Customer` | Dimension | customer_key, customer_type, is_repeated_guest, country |
| `Dim_Channel` | Dimension | channel_key, market_segment, distribution_channel |
| `Dim_Room` | Dimension | room_key, reserved_room_type, assigned_room_type |
| `Dim_Policy` | Junk dimension | policy_key, deposit_type, meal |

Dimensions use **surrogate keys** (1, 2, 3, …); `Dim_Date` uses the date itself as an integer key.

### 2. Setup
Load the cleaned dataset and create an empty SQLite database file in `warehouse/`.

In [36]:
import sqlite3
from pathlib import Path
import pandas as pd

df = pd.read_csv("../data/processed/hotel_bookings_clean.csv", parse_dates=["arrival_date"])
print("Cleaned data:", df.shape)

db_path = Path("../warehouse/hotel_dw.sqlite")
db_path.unlink(missing_ok=True)
con = sqlite3.connect(db_path)
print("Database created:", db_path)

Cleaned data: (86637, 41)
Database created: ../warehouse/hotel_dw.sqlite


### 3. ETL – Extract: staging table
The cleaned bookings are copied into the database as a staging table, `Bookings_Source`.
All warehouse tables are then built from it with SQL, as in the course lab.

In [37]:
df.to_sql("Bookings_Source", con, index=False)
print(pd.read_sql("SELECT COUNT(*) AS rows FROM Bookings_Source", con))

    rows
0  86637


### 4. ETL – Transform & Load: dimension tables
Each dimension stores the **distinct** combinations of its attributes, and each row gets a surrogate key.
`INTEGER PRIMARY KEY` in SQLite numbers the rows automatically (1, 2, 3, …), like `IDENTITY` in SQL Server.

In [38]:
dimensions_sql = """
CREATE TABLE Dim_Date (
    date_key      INTEGER PRIMARY KEY,
    full_date     TEXT,
    year          INTEGER,
    quarter       INTEGER,
    month         INTEGER,
    month_name    TEXT,
    week_number   INTEGER,
    day_of_month  INTEGER,
    day_name      TEXT,
    season        TEXT,
    is_weekend    INTEGER
);
INSERT INTO Dim_Date
SELECT DISTINCT
    CAST(strftime('%Y%m%d', arrival_date) AS INTEGER),
    date(arrival_date),
    arrival_date_year,
    (arrival_month_num - 1) / 3 + 1,
    arrival_month_num,
    arrival_date_month,
    arrival_date_week_number,
    arrival_date_day_of_month,
    CASE strftime('%w', arrival_date)
        WHEN '0' THEN 'Sunday'    WHEN '1' THEN 'Monday'  WHEN '2' THEN 'Tuesday'
        WHEN '3' THEN 'Wednesday' WHEN '4' THEN 'Thursday' WHEN '5' THEN 'Friday'
        ELSE 'Saturday' END,
    season,
    CASE WHEN strftime('%w', arrival_date) IN ('0', '6') THEN 1 ELSE 0 END
FROM Bookings_Source
ORDER BY 1;

CREATE TABLE Dim_Hotel (
    hotel_key  INTEGER PRIMARY KEY,
    hotel_name TEXT NOT NULL
);
INSERT INTO Dim_Hotel (hotel_name)
SELECT DISTINCT hotel FROM Bookings_Source ORDER BY hotel;

CREATE TABLE Dim_Customer (
    customer_key      INTEGER PRIMARY KEY,
    customer_type     TEXT,
    is_repeated_guest INTEGER,
    country           TEXT
);
INSERT INTO Dim_Customer (customer_type, is_repeated_guest, country)
SELECT DISTINCT customer_type, is_repeated_guest, country FROM Bookings_Source
ORDER BY customer_type, is_repeated_guest, country;

CREATE TABLE Dim_Channel (
    channel_key          INTEGER PRIMARY KEY,
    market_segment       TEXT,
    distribution_channel TEXT
);
INSERT INTO Dim_Channel (market_segment, distribution_channel)
SELECT DISTINCT market_segment, distribution_channel FROM Bookings_Source
ORDER BY market_segment, distribution_channel;

CREATE TABLE Dim_Room (
    room_key           INTEGER PRIMARY KEY,
    reserved_room_type TEXT,
    assigned_room_type TEXT
);
INSERT INTO Dim_Room (reserved_room_type, assigned_room_type)
SELECT DISTINCT reserved_room_type, assigned_room_type FROM Bookings_Source
ORDER BY reserved_room_type, assigned_room_type;

CREATE TABLE Dim_Policy (
    policy_key   INTEGER PRIMARY KEY,
    deposit_type TEXT,
    meal         TEXT
);
INSERT INTO Dim_Policy (deposit_type, meal)
SELECT DISTINCT deposit_type, meal FROM Bookings_Source
ORDER BY deposit_type, meal;
"""

con.executescript(dimensions_sql)
con.commit()

In [39]:
for table in ["Dim_Date", "Dim_Hotel", "Dim_Customer", "Dim_Channel", "Dim_Room", "Dim_Policy"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {table}", con)["n"][0]
    print(f"{table:<14} {n:>4} rows")

pd.read_sql("SELECT * FROM Dim_Date LIMIT 5", con)

Dim_Date        793 rows
Dim_Hotel         2 rows
Dim_Customer    442 rows
Dim_Channel      26 rows
Dim_Room         73 rows
Dim_Policy       11 rows


,date_key,full_date,year,quarter,month,month_name,week_number,day_of_month,day_name,season,is_weekend
0,20150701,2015-07-01,2015,3,7,July,27,1,Wednesday,Summer,0
1,20150702,2015-07-02,2015,3,7,July,27,2,Thursday,Summer,0
2,20150703,2015-07-03,2015,3,7,July,27,3,Friday,Summer,0
3,20150704,2015-07-04,2015,3,7,July,27,4,Saturday,Summer,1
4,20150705,2015-07-05,2015,3,7,July,28,5,Sunday,Summer,1


### 5. ETL – Load: fact table
For every booking in the staging table we look up the surrogate key of each dimension with a JOIN,
and store the keys together with the measures. The revenue measures are calculated during loading,
like `Sales_Amount = Quantity × Unit_Price` in the lab:
- `booked_revenue = adr × total_nights`
- `realized_revenue = booked_revenue` if the booking was not canceled, otherwise 0

In [40]:
fact_sql = """
CREATE TABLE Fact_Booking (
    booking_key               INTEGER PRIMARY KEY,
    date_key                  INTEGER NOT NULL REFERENCES Dim_Date(date_key),
    hotel_key                 INTEGER NOT NULL REFERENCES Dim_Hotel(hotel_key),
    customer_key              INTEGER NOT NULL REFERENCES Dim_Customer(customer_key),
    channel_key               INTEGER NOT NULL REFERENCES Dim_Channel(channel_key),
    room_key                  INTEGER NOT NULL REFERENCES Dim_Room(room_key),
    policy_key                INTEGER NOT NULL REFERENCES Dim_Policy(policy_key),
    lead_time                 INTEGER,
    total_nights              INTEGER,
    adults                    INTEGER,
    children                  INTEGER,
    babies                    INTEGER,
    adr                       REAL,
    booked_revenue            REAL,
    realized_revenue          REAL,
    is_canceled               INTEGER,
    booking_changes           INTEGER,
    total_of_special_requests INTEGER
);

INSERT INTO Fact_Booking (
    date_key, hotel_key, customer_key, channel_key, room_key, policy_key,
    lead_time, total_nights, adults, children, babies, adr,
    booked_revenue, realized_revenue, is_canceled, booking_changes, total_of_special_requests
)
SELECT
    CAST(strftime('%Y%m%d', s.arrival_date) AS INTEGER),
    h.hotel_key, cu.customer_key, ch.channel_key, r.room_key, p.policy_key,
    s.lead_time, s.total_nights, s.adults, s.children, s.babies, s.adr,
    s.adr * s.total_nights,
    CASE WHEN s.is_canceled = 0 THEN s.adr * s.total_nights ELSE 0 END,
    s.is_canceled, s.booking_changes, s.total_of_special_requests
FROM Bookings_Source s
JOIN Dim_Hotel    h  ON h.hotel_name = s.hotel
JOIN Dim_Customer cu ON cu.customer_type = s.customer_type
                    AND cu.is_repeated_guest = s.is_repeated_guest
                    AND cu.country = s.country
JOIN Dim_Channel  ch ON ch.market_segment = s.market_segment
                    AND ch.distribution_channel = s.distribution_channel
JOIN Dim_Room     r  ON r.reserved_room_type = s.reserved_room_type
                    AND r.assigned_room_type = s.assigned_room_type
JOIN Dim_Policy   p  ON p.deposit_type = s.deposit_type
                    AND p.meal = s.meal;
"""

con.executescript(fact_sql)
con.commit()
pd.read_sql("SELECT * FROM Fact_Booking LIMIT 5", con)

,booking_key,date_key,hotel_key,customer_key,channel_key,room_key,policy_key,lead_time,total_nights,adults,children,babies,adr,booked_revenue,realized_revenue,is_canceled,booking_changes,total_of_special_requests
0,1,20150701,2,165,10,3,1,7,1,1,0,0,75.0,75.0,75.0,0,0,0
1,2,20150701,2,165,6,1,1,13,1,1,0,0,75.0,75.0,75.0,0,0,0
2,3,20150701,2,165,24,1,1,14,2,2,0,0,98.0,196.0,196.0,0,0,1
3,4,20150701,2,238,10,19,1,0,2,2,0,0,107.0,214.0,214.0,0,0,0
4,5,20150701,2,238,10,19,2,9,2,2,0,0,103.0,206.0,206.0,0,0,1


### 6. ETL validation
Three checks that the fact table was loaded correctly:
1. It has the same number of rows as the source (an INNER JOIN silently drops rows with no match).
2. Its total revenue equals the source total.
3. Every foreign key points to an existing dimension row (no orphan keys).

In [41]:
source_rows = pd.read_sql("SELECT COUNT(*) AS n FROM Bookings_Source", con)["n"][0]
fact_rows = pd.read_sql("SELECT COUNT(*) AS n FROM Fact_Booking", con)["n"][0]

source_rev = pd.read_sql("SELECT SUM(booked_revenue) AS r FROM Bookings_Source", con)["r"][0]
fact_rev = pd.read_sql("SELECT SUM(booked_revenue) AS r FROM Fact_Booking", con)["r"][0]

orphans = pd.read_sql("""
    SELECT COUNT(*) AS n
    FROM Fact_Booking f
    LEFT JOIN Dim_Date d ON f.date_key = d.date_key
    WHERE d.date_key IS NULL
""", con)["n"][0]

print(f"Rows:    source {source_rows:,} | fact {fact_rows:,}")
print(f"Revenue: source €{source_rev:,.2f} | fact €{fact_rev:,.2f}")
print(f"Orphan date keys: {orphans}")

assert source_rows == fact_rows, "Rows were lost in the JOINs"
assert abs(source_rev - fact_rev) < 1, "Revenue does not match"
assert orphans == 0, "Some date keys have no matching row in Dim_Date"
print("\nAll ETL checks passed ✓")

Rows:    source 86,637 | fact 86,637
Revenue: source €34,446,254.93 | fact €34,446,254.93
Orphan date keys: 0

All ETL checks passed ✓


### 7. Analytical SQL
Business questions answered by joining the fact table with its dimensions.

In [42]:
def q(sql):
    return pd.read_sql(sql, con)

#### Q1. Overall totals
How many bookings were canceled, and how much booked revenue was lost?

In [43]:
q("""
SELECT COUNT(*)                                        AS bookings,
       SUM(is_canceled)                                AS canceled,
       ROUND(AVG(is_canceled) * 100, 1)                AS cancel_rate_pct,
       ROUND(SUM(booked_revenue))                      AS booked_revenue,
       ROUND(SUM(realized_revenue))                    AS realized_revenue,
       ROUND(SUM(booked_revenue) - SUM(realized_revenue)) AS lost_revenue
FROM Fact_Booking
""")

,bookings,canceled,cancel_rate_pct,booked_revenue,realized_revenue,lost_revenue
0,86637,23985,27.7,34446255.0,22967612.0,11478643.0


#### Q2. Performance by hotel

In [44]:
q("""
SELECT h.hotel_name,
       COUNT(*)                                             AS bookings,
       ROUND(AVG(f.is_canceled) * 100, 1)                   AS cancel_rate_pct,
       ROUND(SUM(f.booked_revenue))                         AS booked_revenue,
       ROUND(SUM(f.realized_revenue))                       AS realized_revenue,
       ROUND(SUM(f.booked_revenue) - SUM(f.realized_revenue)) AS lost_revenue
FROM Fact_Booking f
JOIN Dim_Hotel h ON f.hotel_key = h.hotel_key
GROUP BY h.hotel_name
""")

,hotel_name,bookings,cancel_rate_pct,booked_revenue,realized_revenue,lost_revenue
0,City Hotel,53042,30.2,18759633.0,12150578.0,6609056.0
1,Resort Hotel,33595,23.7,15686622.0,10817034.0,4869587.0


#### Q3. Where is revenue lost? (by market segment)

In [45]:
q("""
SELECT c.market_segment,
       COUNT(*)                                         AS bookings,
       ROUND(AVG(f.is_canceled) * 100, 1)               AS cancel_rate_pct,
       ROUND(SUM(f.booked_revenue - f.realized_revenue)) AS lost_revenue,
       ROUND(100.0 * SUM(f.booked_revenue - f.realized_revenue)
             / (SELECT SUM(booked_revenue - realized_revenue) FROM Fact_Booking), 1) AS share_of_lost_pct
FROM Fact_Booking f
JOIN Dim_Channel c ON f.channel_key = c.channel_key
GROUP BY c.market_segment
ORDER BY lost_revenue DESC
""")

,market_segment,bookings,cancel_rate_pct,lost_revenue,share_of_lost_pct
0,Online TA,51285,35.5,9292902.0,81.0
1,Direct,11652,14.9,929714.0,8.1
2,Offline TA/TO,13748,14.9,815609.0,7.1
3,Groups,4891,27.2,315620.0,2.7
4,Corporate,4155,12.2,110415.0,1.0
5,Aviation,222,19.8,14065.0,0.1
6,Complementary,682,12.2,270.0,0.0
7,Undefined,2,100.0,48.0,0.0


### 8. OLAP operations
The fact table and its dimensions form a cube (Date × Hotel × Customer × Channel × Room × Policy).
Each operation looks at the cube from a different level or angle.

#### Roll-up: realized revenue from quarter up to year

In [46]:
q("""
SELECT d.year,
       COUNT(*)                           AS bookings,
       ROUND(AVG(f.is_canceled) * 100, 1) AS cancel_rate_pct,
       ROUND(SUM(f.realized_revenue))     AS realized_revenue
FROM Fact_Booking f
JOIN Dim_Date d ON f.date_key = d.date_key
GROUP BY d.year
ORDER BY d.year
""")

,year,bookings,cancel_rate_pct,realized_revenue
0,2015,13142,20.6,3763871.0
1,2016,41966,26.6,10381359.0
2,2017,31529,32.0,8822381.0


#### Drill-down: from year down to quarter

In [47]:
q("""
SELECT d.year, d.quarter,
       COUNT(*)                       AS bookings,
       ROUND(SUM(f.realized_revenue)) AS realized_revenue
FROM Fact_Booking f
JOIN Dim_Date d ON f.date_key = d.date_key
GROUP BY d.year, d.quarter
ORDER BY d.year, d.quarter
""")

,year,quarter,bookings,realized_revenue
0,2015,3,6903,2579783.0
1,2015,4,6239,1184088.0
2,2016,1,8386,1363178.0
3,2016,2,10967,2652685.0
4,2016,3,12049,4261920.0
5,2016,4,10564,2103576.0
6,2017,1,9720,1781779.0
7,2017,2,12920,3507328.0
8,2017,3,8889,3533275.0


#### Slice: Hotel = Resort Hotel, by season

In [48]:
q("""
SELECT d.season,
       COUNT(*)                           AS bookings,
       ROUND(AVG(f.is_canceled) * 100, 1) AS cancel_rate_pct,
       ROUND(AVG(f.adr), 2)               AS avg_adr,
       ROUND(SUM(f.realized_revenue))     AS realized_revenue
FROM Fact_Booking f
JOIN Dim_Date d  ON f.date_key = d.date_key
JOIN Dim_Hotel h ON f.hotel_key = h.hotel_key
WHERE h.hotel_name = 'Resort Hotel'
GROUP BY d.season
ORDER BY CASE d.season WHEN 'Winter' THEN 1 WHEN 'Spring' THEN 2 WHEN 'Summer' THEN 3 ELSE 4 END
""")

,season,bookings,cancel_rate_pct,avg_adr,realized_revenue
0,Winter,6465,17.1,57.19,932258.0
1,Spring,8342,21.8,73.62,1848416.0
2,Summer,11642,31.3,159.66,6276023.0
3,Autumn,7146,19.5,72.91,1760337.0


#### Dice: City Hotel × Online TA × Summer, by customer type

In [49]:
q("""
SELECT cu.customer_type,
       COUNT(*)                           AS bookings,
       ROUND(AVG(f.is_canceled) * 100, 1) AS cancel_rate_pct,
       ROUND(AVG(f.lead_time))            AS avg_lead_time
FROM Fact_Booking f
JOIN Dim_Date d      ON f.date_key = d.date_key
JOIN Dim_Hotel h     ON f.hotel_key = h.hotel_key
JOIN Dim_Channel c   ON f.channel_key = c.channel_key
JOIN Dim_Customer cu ON f.customer_key = cu.customer_key
WHERE h.hotel_name = 'City Hotel'
  AND c.market_segment = 'Online TA'
  AND d.season = 'Summer'
GROUP BY cu.customer_type
ORDER BY bookings DESC
""")

,customer_type,bookings,cancel_rate_pct,avg_lead_time
0,Transient,11092,38.9,103.0
1,Transient-Party,661,13.3,147.0
2,Contract,266,17.3,48.0
3,Group,14,0.0,58.0


#### Pivot: cancellation rate (%) of each market segment, City vs Resort

In [50]:
q("""
SELECT c.market_segment,
       ROUND(AVG(CASE WHEN h.hotel_name = 'City Hotel'   THEN f.is_canceled END) * 100, 1) AS city_hotel,
       ROUND(AVG(CASE WHEN h.hotel_name = 'Resort Hotel' THEN f.is_canceled END) * 100, 1) AS resort_hotel,
       COUNT(*) AS bookings
FROM Fact_Booking f
JOIN Dim_Hotel h   ON f.hotel_key = h.hotel_key
JOIN Dim_Channel c ON f.channel_key = c.channel_key
GROUP BY c.market_segment
HAVING COUNT(*) > 100
ORDER BY bookings DESC
""")

,market_segment,city_hotel,resort_hotel,bookings
0,Online TA,36.2,34.2,51285
1,Offline TA/TO,17.5,12.2,13748
2,Direct,16.6,13.4,11652
3,Groups,33.9,19.5,4891
4,Corporate,11.9,12.6,4155
5,Complementary,10.5,16.4,682
6,Aviation,19.8,NaN,222


#### Ranking: top 10 guest countries by realized revenue

In [51]:
q("""
SELECT RANK() OVER (ORDER BY SUM(f.realized_revenue) DESC) AS revenue_rank,
       cu.country,
       COUNT(*)                           AS bookings,
       ROUND(SUM(f.realized_revenue))     AS realized_revenue,
       ROUND(AVG(f.is_canceled) * 100, 1) AS cancel_rate_pct
FROM Fact_Booking f
JOIN Dim_Customer cu ON f.customer_key = cu.customer_key
GROUP BY cu.country
ORDER BY realized_revenue DESC
LIMIT 10
""")

,revenue_rank,country,bookings,realized_revenue,cancel_rate_pct
0,1,PRT,26863,4849333.0,36.4
1,2,GBR,10400,3724897.0,19.1
2,3,FRA,8813,2736943.0,19.6
3,4,ESP,7228,2019863.0,25.7
4,5,DEU,5385,1622815.0,19.6
5,6,IRL,3014,1173331.0,22.2
6,7,ITA,3056,741903.0,35.1
7,8,BEL,2078,691522.0,19.7
8,9,NLD,1908,579345.0,18.3
9,10,CHE,1564,504590.0,24.6


### 9. Finish the warehouse
The staging table was only needed during ETL, so it is removed. The final warehouse contains the star schema only.

In [52]:
con.execute("DROP TABLE Bookings_Source")
con.commit()
con.execute("VACUUM")

tables = q("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")
for name in tables["name"]:
    n = q(f"SELECT COUNT(*) AS n FROM {name}")["n"][0]
    print(f"{name:<14} {n:>6,} rows")

con.close()
print(f"\nWarehouse saved: {db_path} ({db_path.stat().st_size / 1e6:.1f} MB)")

Dim_Channel        26 rows
Dim_Customer      442 rows
Dim_Date          793 rows
Dim_Hotel           2 rows
Dim_Policy         11 rows
Dim_Room           73 rows
Fact_Booking   86,637 rows

Warehouse saved: ../warehouse/hotel_dw.sqlite (4.3 MB)


ــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــــ


## Summary

**Star schema** (`warehouse/hotel_dw.sqlite`):

| Table | Rows | Role |
| --- | --- | --- |
| `Fact_Booking` | 86,637 | one row per booking: 6 foreign keys + 11 measures |
| `Dim_Date` | 793 | arrival date, with year → quarter → month → day hierarchy |
| `Dim_Customer` | 442 | customer type, repeated guest, country |
| `Dim_Room` | 73 | reserved and assigned room type |
| `Dim_Channel` | 26 | market segment and distribution channel |
| `Dim_Policy` | 11 | deposit type and meal (junk dimension) |
| `Dim_Hotel` | 2 | City Hotel, Resort Hotel |

**ETL checks passed:** same number of rows as the source, identical total revenue, no orphan keys.

**Key findings:**
- Cancellations cost the two hotels **€11.5M, 33.3% of booked revenue**.
- **Online travel agencies account for 81% of the lost revenue**: they are the largest channel (59% of bookings)
  and cancel at 35.5%, more than twice the rate of direct bookings (14.9%).
- City Hotel cancels more than Resort Hotel (30.2% vs 23.7%).
- **Summer (Q3) brings the most revenue every year.** At the Resort Hotel, summer alone brings 58% of realized revenue,
  at almost three times the winter rate, but it is also the season with the most cancellations (31.3%).
- The riskiest segment: individual (Transient) guests booking online for the City Hotel in summer cancel **38.9%** of the time.
- Group bookings cancel almost twice as often at the City Hotel as at the Resort Hotel (33.9% vs 19.5%).

**Caution:** 2015 starts in July and 2017 ends in August, so yearly totals are not directly comparable.

**Next step:** `04_eda.ipynb` – visual exploration of the cleaned data.